In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
from fundus_data_toolkit.functional import open_image
from jppype import Mosaic, vscode_theme

from fundus_odmac_toolkit.models.segmentation import segment as segment_odmac
from fundus_toolkits import FundusData
from fundus_vessels_toolkit.models import segment_av
from fundus_vessels_toolkit.pipelines.avseg_to_tree import GNNAVSegToTree, NaiveAVSegToTree
from fundus_vessels_toolkit.utils.jppype import draw_tree, draw_trees

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

## Load Image and Segment AV, OD, Macula


In [11]:
PATH = Path("/home/gaby/Lab/DATA/Fundus/Fundus-AVSeg/1-images/002_N.png")
IMG = "002_N.png"

fundus_gt = (
    FundusData(image=PATH, av=PATH.parents[1] / "2-av" / PATH.name).crop_to_roi(ensure_square=True).rescale(1500)
)

odmac = segment_odmac(fundus_gt.image.transpose(1, 2, 0) * 255).argmax(axis=0).numpy(force=True)
_ = fundus_gt.update(od=odmac == 1, macula=odmac == 2, inplace=True)


In [ ]:
from turtle import back


fundus = fundus_gt.mutable_copy()
segment_av(fundus)

naive_av2tree = NaiveAVSegToTree()
av2tree = GNNAVSegToTree()

trees = av2tree(fundus)
trees_gt = naive_av2tree(fundus_gt)


m = Mosaic(2, cols_titles=["Predicted", "Ground Truth"], cell_height=800, background=fundus.image)
fundus_gt.draw(view=m[0])
# draw_trees(trees, view=m[0])
# fundus_gt.draw(view=m[1])
draw_trees(trees_gt, view=m[1])
m

GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…

In [26]:
from fundus_vessels_toolkit.vparameters import parametrize_branches

parametrize_branches(trees[0], fundus_gt)

,branch,arc,arc_bspline,chord,norm_dist_od,norm_coord_y,norm_coord_x,dist_od,dist_macula,mean_calibre,...,mean_curvature,std_curvature,length_diameter_ratio,τHart,τGrisan,τTrucco,n_curvatures_roots,τHart_bspline,τGrisan_bspline,n_bspline
branch,,,,,,,,,,,,,,,,,,,,,
0,0,59.012193,56.391835,54.037024,2.511564,-1.099414,0.361796,677.177708,604.256294,11.617961,...,-0.013568,0.009365,5.079393,0.092070,0.000000,0.049581,1,0.043578,0.001736,3
1,1,128.710678,119.689751,112.044634,4.460173,-1.327977,1.367660,1115.340197,667.029359,6.429066,...,0.006472,0.013100,20.020120,0.148745,0.004700,0.061138,3,0.068233,0.000562,2
2,2,303.620685,267.686068,239.616360,4.357489,-1.019562,1.563843,1092.250823,565.224666,5.045134,...,-0.007025,0.015665,60.180901,0.267112,0.014424,0.082284,7,0.117144,0.003488,6
3,3,108.740115,102.229937,98.412398,4.618668,-1.271567,1.501027,1150.979210,669.068435,5.476042,...,0.002400,0.016745,19.857428,0.104943,0.001886,0.061883,2,0.038791,0.000669,2
4,4,66.384776,62.548174,62.369865,4.232594,-1.310397,1.261372,1064.166983,637.793881,4.579334,...,0.002818,0.009601,14.496601,0.064373,0.007345,0.031355,3,0.002859,0.000179,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,94,64.083261,59.274550,60.216277,3.979508,-0.448135,1.662230,1007.258378,422.180934,5.400812,...,-0.006598,0.009538,11.865487,0.064218,0.003152,0.040149,2,-0.015639,0.000338,3
95,95,74.911688,68.258280,68.249542,2.220872,-0.575946,0.872790,611.813055,191.886960,5.248122,...,0.000212,0.023428,14.274000,0.097615,0.000000,0.082394,1,0.000128,0.000335,3
96,96,102.112698,95.340672,95.566731,4.187941,-0.701246,1.659615,1054.126327,480.937226,10.637055,...,0.001910,0.004564,9.599715,0.068496,0.005301,0.017860,3,-0.002365,0.000130,2


## Compute AV topological maps


In [18]:
from fundus_vessels_toolkit.segment_to_graph.av_map_fixing import TopologicalLabel, rasterize_tree_topology

topo_maps = [rasterize_tree_topology(tree, expand_labels_by=10, bridge_gap_smaller_than=50) for tree in trees_gt]
(art_labels, art_topo), (vei_labels, vei_topo) = topo_maps

m = Mosaic(
    (2, 3),
    cols_titles=["VTree", "Branch labels", "Topology map"],
    rows_titles=["Art.", "Vein"],
    cell_height=800,
    background=fundus.image,
)
draw_tree(trees_gt[0], view=m[0, 0], artery=True, edge_labels=False)
m[0, 0].add_label(fundus_gt.av == 1, colormap="red", opacity=0.2)
m[0, 1].add_image(TopologicalLabel.map_to_rgb(art_labels))
m[0, 2].add_image(np.repeat(art_topo[:, :, None], 3, axis=2))
fundus_gt.draw(view=m[1, 0])
draw_tree(trees_gt[1], view=m[1, 0], artery=False, edge_labels=False)
m[1, 1].add_image(TopologicalLabel.map_to_rgb(vei_labels))
m[1, 2].add_image(np.repeat(vei_topo[:, :, None], 3, axis=2))
m

GridBox(children=(HTML(value='<span/>'), HTML(value='<h3 style="text-align: center;">VTree</h3>'), HTML(value=…

In [23]:
m = Mosaic(
    3,
    cell_height=800,
    background=fundus.image,
)
fundus_gt.draw(view=m[0])
draw_trees(trees, view=m[0])
m[1].add_image(TopologicalLabel.map_to_rgb(art_labels))
draw_tree(trees[0], view=m[1], artery=True)
m[2].add_image(np.repeat(vei_topo[:, :, None], 3, axis=2))
draw_tree(trees[1], view=m[2], artery=False)
m

GridBox(children=(View2D(linkedTransformGroup='f7434245f171497fa561ce453edf8a80'), View2D(linkedTransformGroup…

In [16]:
from fundus_vessels_toolkit.segment_to_graph.models.training import deteriorate_segmentation

fundus_gt2 = fundus_gt.update(av=deteriorate_segmentation(fundus_gt))
trees_gt2 = naive_av2tree(fundus_gt2)

m = Mosaic(2, cols_titles=["Predicted", "Ground Truth"], cell_height=800)
fundus_gt.draw(view=m[0])
draw_trees(trees_gt, edge="skeleton", view=m[0])
fundus_gt2.draw(view=m[1])
draw_trees(trees_gt2, view=m[1])
m

Dropped 1 segments of size 20 from branch 1
Dropped 1 segments of size 44 from branch 2
Dropped 0 segments of size 39 from branch 4
Dropped entire branch 6
Dropped 0 segments of size 32 from branch 7
Dropped 1 segments of size 34 from branch 10
Dropped 0 segments of size 31 from branch 11
Dropped 0 segments of size 31 from branch 12
Dropped 1 segments of size 55 from branch 14
Dropped 1 segments of size 55 from branch 19
Dropped 1 segments of size 16 from branch 24
Dropped 1 segments of size 46 from branch 25
Dropped 1 segments of size 44 from branch 27
Dropped 1 segments of size 30 from branch 29
Dropped 1 segments of size 28 from branch 34
Dropped 2 segments of size 56 from branch 38
Dropped 0 segments of size 57 from branch 39
Dropped 0 segments of size 15 from branch 40
Dropped 1 segments of size 64 from branch 41
Dropped 1 segments of size 24 from branch 43
Dropped 1 segments of size 16 from branch 45
Dropped 1 segments of size 45 from branch 48
Dropped 2 segments of size 36 from 

GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…